# Natural overlapping audio: complete reproducible workflow

Use authentic MAESTRO Real recordings and human soft temporal annotations. This notebook calls production modules for setup, features, learned training, saved inference and aggregate plots. All outputs are intentionally empty; recorded cloud/CPU evidence is separate from optional new runs. No encoded source, invented measurements or automatic observed-test rescore.

Install `python -m pip install -e ".[dev,transfer]"`. MAESTRO has custom experimental/noncommercial terms; retain `docs/MAESTRO-LICENSE.txt`. Data, weights and derived artifacts remain ignored. Full audio download is about 2.6 GB. PANNs extraction/inference requires substantial memory; prefer private cloud compute on an 8 GB host.

In [ ]:
from pathlib import Path
import sys, json, subprocess
from datetime import datetime, timezone
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
print('Repository:', ROOT)
print('Fresh run ID:', RUN_ID)
sys.path.insert(0, str(ROOT / 'src'))


## 1. Real data and licensing
Fetching is opt-in, source checksums/provenance are enforced by the real adapter. No synthetic replacement appears when files are missing. Read Report.md and docs/transfer-evaluation.md before interpreting the measurements.

In [ ]:
FETCH_ANNOTATIONS = False
FETCH_FULL_AUDIO = False
MAESTRO = ROOT / 'data' / 'external' / 'maestro'
from overlap_audio.fetch_maestro import fetch
if FETCH_ANNOTATIONS or FETCH_FULL_AUDIO:
    provenance = fetch(MAESTRO, audio=FETCH_FULL_AUDIO)
    print(provenance['license'])
print('Local real recordings:', len(list(MAESTRO.rglob('*.wav'))))


## 2. Frozen PANNs features and source contracts
Use official Cnn14 AudioSet weights SHA256 `0dc499e40e9761ef5ea061ffc77697697f277f6a960894903df3ada000e34b31`, native 32 kHz and exact five-second windows. The production adapter retains recording split, second alignment, masks and soft labels. Test annotations may be staged, but no test scores or threshold selection occur in feature extraction. Source and weights remain private.

In [ ]:
RUN_FEATURE_EXTRACTION = False
WEIGHTS = ROOT / 'data' / 'external' / 'panns' / 'Cnn14.pth'
CACHE = ROOT / 'data' / 'features' / 'five-second-cache'
DEVICE = 'cpu'  # choose available cloud GPU only when authorized; never silently assume it
if RUN_FEATURE_EXTRACTION:
    from overlap_audio.transfer import prepare
    CACHE = ROOT / 'data' / 'features' / ('notebook-' + RUN_ID)
    prepare(MAESTRO, CACHE, WEIGHTS, device=DEVICE, context_seconds=5)
metadata_path = CACHE / 'metadata.json'
if metadata_path.exists():
    metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
    print({k: metadata[k] for k in ('context_seconds', 'classes', 'assignment')})
else:
    print('Authentic private cache absent; no substitute features are fabricated.')


## 3. Opt-in actual learned training
Root train.py uses only train and validation. Three fixed seeds train a prespecified configuration with masked soft BCE, train-only normalization and positive weights. Every run path must be fresh. This cell never opens test or changes the historical production winner.

In [ ]:
RUN_TRAINING = False
TRAINED = ROOT / 'models' / ('notebook-' + RUN_ID)
if RUN_TRAINING:
    from overlap_audio.entrypoints import train_only
    result = train_only(CACHE, TRAINED, config='slower', seeds=[42, 1337, 2026], epochs=80, device=DEVICE)
    print(result['validation'])
else:
    print('Training disabled. Historical comparisons below read committed aggregates.')


## 4. Recorded validation experiments and learning curves
All twelve actual V6 fits are retained even though none beat V5 on validation. Individual seeds are diagnostics, not a menu for selecting a lucky seed. Centered smoothing requires future context and resets at recording boundaries/gaps.

In [ ]:
study = json.loads((ROOT / 'docs' / 'five-second-learned-v6-selection.json').read_text(encoding='utf-8'))
comparison = [(r['config']['name'], r['equal_seed_ensemble_validation_ap'], r['individual_seed_std_ap']) for r in study['candidates']]
print(comparison)
print('Retained prior:', not study['improved'], 'validation AP:', study['selected_validation_ap'])


In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar([r[0] for r in comparison], [r[1] for r in comparison])
ax.axhline(study['prior_validation_ap'], color='black', linestyle='--', label='Retained V5 policy')
ax.set(ylabel='Supported-class validation macro AP', title='Actual learned comparisons'); ax.legend(); plt.tight_layout()
fig, ax = plt.subplots(figsize=(9, 4))
for candidate in study['candidates']:
    for fit in candidate['fits']:
        ax.plot([r['epoch'] for r in fit['history']], [r['validation_ap'] for r in fit['history']], alpha=.6, label=candidate['config']['name'] + '/' + str(fit['seed']))
ax.set(xlabel='Epoch', ylabel='Validation macro AP'); ax.legend(fontsize=7, ncol=2); plt.tight_layout()


## 5. Saved policy evaluation on validation only
This optional cell uses the checkpoint inference path and validates cache contracts. It does not train, tune, overwrite historical artifacts or automatically rescore observed test. Test evaluation is deliberately absent from the notebook.

In [ ]:
RUN_VALIDATION_REPLAY = False
CHECKPOINT = ROOT / 'models' / 'postprocess-v5' / 'selected.pt'
if RUN_VALIDATION_REPLAY:
    from overlap_audio.entrypoints import evaluate
    replay = evaluate(CHECKPOINT, {5: CACHE}, ROOT / 'data' / 'processed' / ('validation-' + RUN_ID + '.json'))
    print(replay['metrics'])


## 6. Actual saved-model waveform inference
Choose an actual licensed local recording. Production inference verifies checkpoint schema and encoder hash, preserves recording/time boundaries and reports lookahead. No labels are passed. Run on cloud compute if local memory is limited. The recorded V5 waveform proof is a mechanics check, not a new accuracy claim.

In [ ]:
RUN_WAVEFORM_INFERENCE = False
AUDIO = next(iter(MAESTRO.rglob('*.wav')), None)
if RUN_WAVEFORM_INFERENCE:
    if AUDIO is None: raise FileNotFoundError('Provide an actual licensed recording')
    from overlap_audio.postprocess import infer
    inference = infer(CHECKPOINT, WEIGHTS, AUDIO, ROOT / 'data' / 'processed' / ('waveform-' + RUN_ID + '.json'), DEVICE)
    print('Seconds predicted:', len(inference['observations']), 'lookahead:', inference['lookahead_seconds'])
else:
    proof = json.loads((ROOT / 'docs' / 'postprocess-v5-actual-inference-proof.json').read_text(encoding='utf-8'))
    print('Recorded cloud inference proof:', proof)


## 7. Recorded observed-test result and rare-class limits
These are already released aggregate results, not a fresh independent test. V5 aggregate was opened after V6 protocol freeze but before V6 selection completed. No V6 grid/promotion/stopping changed in response. Unsupported classes have AP undefined, not zero. Rare coffee/cart supervision and class-specific regressions remain visible.

In [ ]:
report = json.loads((ROOT / 'docs' / 'postprocess-v5-observed-holdout.json').read_text(encoding='utf-8'))
print({k: report['selected'][k] for k in ('macro_ap_supported', 'macro_f1_all_observed_classes', 'macro_f1_supported', 'micro_f1', 'unsupported_classes')})
rows = [r for r in report['selected']['per_class'] if r['ap'] is not None]
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar([r['class'] for r in rows], [r['ap'] for r in rows]); ax.tick_params(axis='x', rotation=70)
ax.set(ylabel='Observed-test AP', title='Supported classes only; no test-guided selection'); plt.tight_layout()
